# 2 — Patterns

Loads the tables built by **notebook 1** and looks for structure. **It reads no logs** — if this
notebook runs, the dataset is genuinely self-contained.

The order is deliberate:

1. **What is in the dataset** — sessions per mouse per world, how long they run, how many trials.
2. **The whole world, pooled** — every session of one world in a single answer.
3. **Per-session lines** — is there any shape over days?
4. **Blocks** — only now, and only if step 3 suggests something worth testing.
5. **First half vs second half.**

Steps 1–3 draw **no trend lines and no verdicts**. Picking a binning after seeing a curve is how a
pattern gets manufactured, so the un-binned answer comes first and `BLOCK_SIZE` is chosen in step 4,
after you have looked.

## Load ← YOU SET THIS

In [ ]:
MAIN_DIR = '/mnt/server/data'
PIPELINE_DIR = None
# =============================================================================

import sys, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'performance'))

import session_index as sidx, perf_from_log as pfl, build_log_df as bl, plot_patterns as pp
sidx = importlib.reload(sidx); pfl = importlib.reload(pfl)
bl = importlib.reload(bl); pp = importlib.reload(pp)

OUT = MAIN_DIR / 'df_log'
df_sessions = bl.load(OUT / 'df_sessions.pkl')
df_trials   = bl.load(OUT / 'df_trials.pkl')
print(f'\n{len(df_sessions)} sessions, {len(df_trials)} trials, '
      f'{df_sessions.mouse.nunique()} animal(s)')

## 1 — What is in the dataset

Before any performance question. A `D` curve cannot be read without knowing that one point is a
40-minute session of 80 trials and the next is a 10-minute session of 12 — panels (d) and (e) are
what tell you whether two points are comparable at all.

In [ ]:
fig = pp.overview(df_sessions, df_trials)
plt.show()

display(df_sessions.groupby(['mouse', 'world_id', 'task'])
        .agg(sessions=('session', 'count'), trials=('n_trials_total', 'sum'),
             minutes=('wall_min', 'sum'), first=('day', 'min'), last=('day', 'max'))
        .round(1))

## 2 — Choose what to look at ← YOU SET THIS

Chosen **after** the table above, not before. A world fixes the viewport, and therefore the chance
baseline, so sessions from different worlds are not directly comparable.

In [ ]:
ANIMAL = None        # None = every animal in the dataset, or e.g. 'JPAS_0168'
WORLD  = None        # None = every world, or e.g. 'W4'
TASK   = None        # None = every protocol, or e.g. 'banish_multiplier'
USABLE_ONLY = True   # drop sessions flagged unusable, and the earlier of any doubled day
# =============================================================================

X = df_sessions.copy()
if ANIMAL: X = X[X.mouse == ANIMAL]
if WORLD:  X = X[X.world_id == WORLD]
if TASK:   X = X[X.task == TASK]
if USABLE_ONLY:
    X = X[X.use & X.keep_of_day & X.perf_error.eq('')]
X = X.sort_values(['day', 'time']).reset_index(drop=True)

T = df_trials[df_trials.session.isin(X.session)]
print(f'{len(X)} session(s), {len(T)} trial(s)   |   '
      f'{sorted(X.mouse.unique())}   {sorted(X.world_id.unique())}   {sorted(X.task.unique())}')
if X.world_id.nunique() > 1:
    print('\n  !! more than one WORLD selected: ' + ', '.join(sorted(X.world_id.unique())))
    print('     A world fixes the viewport, so each has its own chance baseline. Set WORLD.')
if X.task.nunique() > 1:
    print('\n  !! more than one PROTOCOL selected: ' + ', '.join(sorted(X.task.unique())))
    for t in sorted(X.task.unique()):
        print(f'        {t:20s} {pfl.TASK_DESCRIPTION.get(t, "")}')
    print('     These are DIFFERENT EXPERIMENTS -- different icons, different reward units,')
    print('     different chance baseline. Step 3 will refuse to pool them. Set TASK.')
display(X[['session', 'day', 'task', 'n_trials', 'acc', 'chance', 'D', 'p', 'drops']].round(3))

## 3 — The whole world, pooled

Every selected session in **one** answer, with no binning at all.

**CONFLICT** trials are the ones where the board put the hazard *nearer*, so proximity has to be
overridden — that is the number that must rise with learning. **CONTROL** trials had the reward
nearer and should stay high. The two must **separate**: if both move together he has merely stopped
following proximity, which is not the same as recognising the icon.

In [ ]:
B_all = sidx.blocks_of(X, size=None)        # size=None -> one block, every session
display(B_all[['block', 'n_sessions', 'k_conflict', 'n_conflict', 'conflict_p',
               'k_control', 'n_control', 'control_p', 'D', 'acc']].round(3))

fig = pp.pooled_criterion(B_all, title=f'{" / ".join(sorted(X.mouse.unique()))} — '
                                       f'{" / ".join(sorted(X.world_id.unique()))}')
plt.show()

## 4 — Per-session pattern

One point per session, every measure, on a shared x-axis so a bump in `D` can be read against what
throughput, trial count and session length were doing the same day.

**Nothing is fitted and nothing is concluded.** A session with few scored trials has a wide interval
— check the `n` panel before believing any single point.

In [ ]:
fig = pp.session_lines(X, title=f'{len(X)} sessions — nothing fitted')
plt.show()

## 5 — Blocks ← ONLY IF STEP 4 SUGGESTS SOMETHING

Pooling raises the trial count per point, which narrows the intervals — but the bin width should be
chosen because of what step 4 showed, not before looking. One session carries only ~10–15 conflict
trials (±0.25), so a single session is uninformative on its own; about a week is the usual unit.

In [ ]:
BLOCK_SIZE = 5      # sessions per block -- about a working week
# =============================================================================

B = sidx.blocks_of(X, size=BLOCK_SIZE)
display(B[['block', 'label', 'n_sessions', 'conflict_p', 'conflict_lo', 'conflict_hi',
           'control_p', 'D']].round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(B))
a = ax[0]
a.errorbar(x, B.conflict_p, yerr=[(B.conflict_p - B.conflict_lo).clip(lower=0),
                                  (B.conflict_hi - B.conflict_p).clip(lower=0)],
           fmt='o-', color='#e67e22', capsize=4, lw=2, label='CONFLICT (must rise)')
a.errorbar(x, B.control_p, yerr=[(B.control_p - B.control_lo).clip(lower=0),
                                 (B.control_hi - B.control_p).clip(lower=0)],
           fmt='s--', color='#7f8c8d', capsize=4, label='CONTROL (stays high)')
a.axhline(.5, ls=':', color='k'); a.set_ylim(0, 1)
a.set_xticks(x); a.set_xticklabels(B.label, fontsize=7)
a.set_ylabel('P(collected the reward)'); a.legend(fontsize=8)
a.set_title('THE CRITERION, per block', fontweight='bold')
a.grid(alpha=.2)

a = ax[1]
a.bar(x, B.control_p - B.conflict_p, color='#c0392b')
a.set_xticks(x); a.set_xticklabels(B.label, fontsize=7)
a.set_ylabel('control − conflict')
a.set_title('THE GAP per block\n(shrinking = overriding proximity to avoid the hazard)',
            fontweight='bold')
a.grid(alpha=.2, axis='y')
plt.tight_layout(); plt.show()

## 6 — First half vs second half

A **fixed** split, so it is not chosen by eye. Two different questions kept apart: **within session**
(does he fade during a session?) and **across days** (did he change over training?). Within-session
halves are pooled per session so one long session cannot dominate both sides. Escapes are excluded —
they offer no reward-vs-hazard choice.

The numbers and their intervals are printed; read them yourself.

In [ ]:
fig, HALVES, PERSESS = pp.half_split(T, X)
plt.show()

for k, d in HALVES.items():
    if 'within' in k:
        print(f"{k:15s} P(reward) = {d['p']:.3f}   n = {d['n']:4d}   "
              f"95% confidence interval [{d['lo']:.3f}, {d['hi']:.3f}]")
    else:
        print(f"{k:15s} mean D    = {d['D']:+.3f}   over {d['n']} session(s)")
print('\nOverlapping intervals mean the difference is not resolved by this much data.')
PERSESS